# 04 Statistics Demo

This notebook demonstrates how the corrected and uncorrected statistical result logic from the original MATLAB fNIRS pipeline is translated into Python.

The MATLAB pipeline reports:

1. uncorrected results using p < .05
2. FWE-corrected results using p < .05 / 32
3. significant positive and negative channels for each contrast

This notebook uses simulated statistical results only. No real subject data are included.


In [1]:
from pathlib import Path
import sys
import importlib.util

import numpy as np
import pandas as pd

# Locate project root
cwd = Path.cwd()
project_root = cwd if (cwd / 'src').exists() else cwd.parent

# Load local src/statistics.py safely
stats_path = project_root / 'src' / 'statistics.py'
spec = importlib.util.spec_from_file_location('fnirs_statistics', stats_path)
fnirs_statistics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fnirs_statistics)

get_fwe_threshold = fnirs_statistics.get_fwe_threshold
add_significance_flags = fnirs_statistics.add_significance_flags
get_uncorrected_results = fnirs_statistics.get_uncorrected_results
get_fwe_corrected_results = fnirs_statistics.get_fwe_corrected_results
split_positive_negative = fnirs_statistics.split_positive_negative


## Step 1: Define the FWE threshold

The original MATLAB pipeline uses:

`p < .05 / 32`

This is a Bonferroni-style family-wise error correction.


In [2]:
fwe_threshold = get_fwe_threshold(alpha=0.05, n_tests=32)
fwe_threshold


0.0015625

## Step 2: Create a simulated channel-wise results table

In the real pipeline, this table should come from the group-level GLM output.

For now, we create simulated results to verify the statistical filtering logic.


In [3]:
np.random.seed(42)

channels = [f'Ch{i:02d}' for i in range(1, 33)]
contrasts = [
    'G4_6_MA_minus_Control',
    'G1_3_MA_minus_Control',
    'Group_difference_MA_minus_Control'
]

rows = []
for contrast in contrasts:
    for ch in channels:
        rows.append({
            'channel': ch,
            'contrast': contrast,
            'effect': np.random.normal(loc=0, scale=0.25),
            'p_value': np.random.uniform(0, 0.10)
        })

results_df = pd.DataFrame(rows)

# Manually create a few very small p-values so that the FWE logic can be demonstrated
results_df.loc[0, ['effect', 'p_value']] = [0.55, 0.0008]
results_df.loc[10, ['effect', 'p_value']] = [-0.48, 0.0010]
results_df.loc[40, ['effect', 'p_value']] = [0.35, 0.0200]

results_df.head()


,channel,contrast,effect,p_value
0,Ch01,G4_6_MA_minus_Control,0.550000,0.000800
1,Ch02,G4_6_MA_minus_Control,-0.034566,0.059866
2,Ch03,G4_6_MA_minus_Control,-0.058538,0.005808
3,Ch04,G4_6_MA_minus_Control,-0.058534,0.086618
4,Ch05,G4_6_MA_minus_Control,0.394803,0.002058


## Step 3: Add uncorrected and FWE-corrected significance flags


In [4]:
results_with_flags = add_significance_flags(
    results_df,
    p_col='p_value',
    effect_col='effect',
    alpha=0.05,
    n_tests=32
)

results_with_flags.head()


,channel,contrast,effect,p_value,significant_uncorrected,significant_fwe,direction
0,Ch01,G4_6_MA_minus_Control,0.550000,0.000800,True,True,positive
1,Ch02,G4_6_MA_minus_Control,-0.034566,0.059866,False,False,negative
2,Ch03,G4_6_MA_minus_Control,-0.058538,0.005808,True,False,negative
3,Ch04,G4_6_MA_minus_Control,-0.058534,0.086618,False,False,negative
4,Ch05,G4_6_MA_minus_Control,0.394803,0.002058,True,False,positive


## Step 4: Extract uncorrected significant results

These are channels with p < .05 before multiple-comparison correction.


In [5]:
uncorrected_results = get_uncorrected_results(
    results_with_flags,
    p_col='p_value',
    alpha=0.05
)

uncorrected_results.head(10)


,channel,contrast,effect,p_value,significant_uncorrected,significant_fwe,direction
0,Ch01,G4_6_MA_minus_Control,0.550000,0.000800,True,True,positive
2,Ch03,G4_6_MA_minus_Control,-0.058538,0.005808,True,False,negative
4,Ch05,G4_6_MA_minus_Control,0.394803,0.002058,True,False,positive
6,Ch07,G4_6_MA_minus_Control,-0.117369,0.018182,True,False,negative
7,Ch08,G4_6_MA_minus_Control,0.135640,0.018340,True,False,positive
8,Ch09,G4_6_MA_minus_Control,0.060491,0.043195,True,False,positive
9,Ch10,G4_6_MA_minus_Control,-0.478320,0.029123,True,False,negative
10,Ch11,G4_6_MA_minus_Control,-0.480000,0.001000,True,True,negative
11,Ch12,G4_6_MA_minus_Control,0.078562,0.036636,True,False,positive
12,Ch13,G4_6_MA_minus_Control,0.366412,0.019967,True,False,positive


## Step 5: Extract FWE-corrected significant results

These are channels with p < .05 / 32.


In [6]:
fwe_results = get_fwe_corrected_results(
    results_with_flags,
    p_col='p_value',
    alpha=0.05,
    n_tests=32
)

fwe_results


,channel,contrast,effect,p_value,significant_uncorrected,significant_fwe,direction
0,Ch01,G4_6_MA_minus_Control,0.550000,0.000800,True,True,positive
10,Ch11,G4_6_MA_minus_Control,-0.480000,0.001000,True,True,negative
32,Ch01,G1_3_MA_minus_Control,-0.169231,0.000552,True,True,negative
58,Ch27,G1_3_MA_minus_Control,0.074030,0.000695,True,True,positive
94,Ch31,Group_difference_MA_minus_Control,0.073268,0.000506,True,True,positive


## Step 6: Split positive and negative significant channels

The MATLAB pipeline extracts significant positive and negative channels separately.


In [7]:
positive_fwe, negative_fwe = split_positive_negative(
    fwe_results,
    effect_col='effect'
)

positive_fwe


,channel,contrast,effect,p_value,significant_uncorrected,significant_fwe,direction
0,Ch01,G4_6_MA_minus_Control,0.550000,0.000800,True,True,positive
58,Ch27,G1_3_MA_minus_Control,0.074030,0.000695,True,True,positive
94,Ch31,Group_difference_MA_minus_Control,0.073268,0.000506,True,True,positive


In [8]:
negative_fwe


,channel,contrast,effect,p_value,significant_uncorrected,significant_fwe,direction
10,Ch11,G4_6_MA_minus_Control,-0.480000,0.001000,True,True,negative
32,Ch01,G1_3_MA_minus_Control,-0.169231,0.000552,True,True,negative


## Summary

This notebook verifies that the Python pipeline can reproduce the statistical filtering logic used in the MATLAB pipeline:

- uncorrected threshold: p < .05
- FWE-corrected threshold: p < .05 / 32
- extraction of significant positive and negative channels

The next step is to connect this logic to real group-level GLM output or to a processed anonymized result table.
